## Direct Step Functions Integration

Welcome to the final lesson of this course on building serverless applications! So far, you have learned how to use AWS Lambda, API Gateway, and Step Functions to create and deploy serverless workflows. In this lesson, we will take your skills a step further by showing you how to use API Gateway to start Step Function executions directly — without needing a Lambda function in between.

This approach can make your applications faster and more cost-effective, since you avoid the extra step and cost of running Lambda just to start a workflow. You will learn how to use mapping templates to control the request and response, validate input, handle errors, and even route to different workflows — all using the Velocity Template Language (VTL).

By the end of this lesson, you will be able to build a robust API that starts Step Function executions with dynamic input and strong validation, all without writing any Lambda code.

---

## Quick Recall: Mapping Templates and Direct Integrations

Before we dive in, let's quickly remind ourselves what mapping templates are and why they matter.

A mapping template in API Gateway is a way to transform the incoming request or outgoing response. It lets you change the shape of the data, add or remove fields, and even add logic — using a language called Velocity Template Language (VTL).

In previous lessons, you may have seen mapping templates used to format data for Lambda functions. In this lesson, we will use them to talk directly to Step Functions, letting us control exactly how the API request is turned into a Step Function execution.

---

## Generating Unique Execution Names with VTL

When starting a Step Function execution, you often want each execution to have a unique name. This helps you track and manage each workflow run.

Let's see how to do this using a request mapping template in API Gateway.

First, we use the `$util.autoId()` function in VTL to generate a random string. We can combine this with a prefix, like `"order-"`, to create a unique execution name.

Here's how you can set this up in your mapping template:

```vtl
#set($inputRoot = $util.parseJson($input.body))
{
  "stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine",
  "name": "order-$util.autoId()",
  "input": "$util.escapeJavaScript($input.body)"
}
```

**Explanation:**

* `#set($inputRoot = $util.parseJson($input.body))` parses the incoming JSON body so you can access its fields.
* `"name": "order-$util.autoId()"` creates a unique execution name like `order-abc123xyz`.
* `"input": "$util.escapeJavaScript($input.body)"` passes the original request body as input to the Step Function.

**Example Output:**

```json
{
  "stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine",
  "name": "order-7f3b2c1a",
  "input": "{\"customerId\": \"12345\", \"product\": \"laptop\", \"quantity\": 2}"
}
```

This ensures every execution has a unique name, which is important for tracking and debugging.

---

## Input Validation in Request Mapping Templates

It's important to make sure the incoming request has all the required fields before starting a workflow. For example, you might require a `customerId` in the request body.

You can use VTL's `#if` directive to check for required fields and return an error if something is missing.

Here's how you can add validation to your mapping template:

```vtl
#set($inputRoot = $util.parseJson($input.body))
#if(!$inputRoot.customerId || $inputRoot.customerId == "")
  #set($context.responseOverride.status = 400)
  #set($context.responseOverride.header.Content-Type = "application/json")
  {
    "error": "Missing required field: customerId"
  }
#else
  {
    "stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine",
    "name": "order-$util.autoId()",
    "input": "$util.escapeJavaScript($input.body)"
  }
#end
```

**Explanation:**

* `#if(!$inputRoot.customerId || $inputRoot.customerId == "")` checks if `customerId` is missing or empty.
* If it is, we set the response status to 400 and return an error message.
* If not, we proceed to start the Step Function execution as before.

**Example Output (when `customerId` is missing):**

```json
{
  "error": "Missing required field: customerId"
}
```

This helps prevent invalid requests from starting workflows, saving resources and making your API more reliable.

---

## Transforming Step Functions Responses

When the Step Function execution starts, the response from AWS is not always user-friendly. You can use a response mapping template to transform the response and add helpful information.

For example, you might want to include:

* The execution ARN
* The execution name
* A timestamp for when the execution started
* A status field

Here's how you can do this in your response mapping template:

```vtl
#set($inputRoot = $util.parseJson($input.body))
{
  "execution_arn": "$inputRoot.executionArn",
  "execution_name": "$inputRoot.name",
  "started_at": "$util.time.nowISO8601()",
  "status": "RUNNING"
}
```

**Explanation:**

* `$inputRoot.executionArn` and `$inputRoot.name` come from the Step Functions response.
* `$util.time.nowISO8601()` adds the current timestamp in a standard format.
* `"status": "RUNNING"` gives a clear status for the client.

**Example Output:**

```json
{
  "execution_arn": "arn:aws:states:us-east-1:123456789012:execution:OrderProcessingStateMachine:order-7f3b2c1a",
  "execution_name": "order-7f3b2c1a",
  "started_at": "2024-06-01T12:34:56Z",
  "status": "RUNNING"
}
```

This makes your API responses easier to understand and use.

---

## Error Handling for Step Functions Integration

Sometimes, starting a Step Function execution can fail. For example, the state machine might not exist, or you might try to use a duplicate execution name.

You can handle these errors in your API by mapping them to appropriate HTTP status codes and messages.

Here's how you can set up error responses in your API Gateway integration:

```yaml
responses:
  default:
    statusCode: 202
    responseTemplates:
      application/json: |
        #set($inputRoot = $util.parseJson($input.body))
        {
          "execution_arn": "$inputRoot.executionArn",
          "execution_name": "$inputRoot.name",
          "started_at": "$util.time.nowISO8601()",
          "status": "RUNNING"
        }
  4\d{2}:
    statusCode: 400
    responseTemplates:
      application/json: |
        {
          "error": "Invalid request parameters"
        }
  5\d{2}:
    statusCode: 500
    responseTemplates:
      application/json: |
        {
          "error": "Internal server error",
          "message": "$util.escapeJavaScript($input.body)"
        }
```

**Explanation:**

* The `default` response handles successful executions.
* The `4\d{2}` pattern matches client errors (like validation failures) and returns a 400 status.
* The `5\d{2}` pattern matches server errors and returns a 500 status with an error message.

**Example Output (for a duplicate execution name):**

```json
{
  "error": "Invalid request parameters"
}
```

**Example Output (for an internal error):**

```json
{
  "error": "Internal server error",
  "message": "ExecutionAlreadyExists: An execution with the same name already exists."
}
```

This makes your API more robust and user-friendly by providing clear error messages.

---

## Dynamic Workflow Routing with Request Templates

Sometimes, you want your API to start different workflows based on a parameter in the request. For example, you might have different workflows for orders, refunds, or inventory updates.

You can read a `workflow_type` parameter from the request body, validate it, and use it to customize the Step Function input and execution name.

Here's how you can do this in your request mapping template:

```vtl
#set($inputRoot = $util.parseJson($input.body))
#set($workflowType = "order")
#if($inputRoot.workflow_type && $inputRoot.workflow_type != "")
  #if($inputRoot.workflow_type == "order" || $inputRoot.workflow_type == "refund" || $inputRoot.workflow_type == "inventory")
    #set($workflowType = $inputRoot.workflow_type)
  #end
#end
#if(!$inputRoot.customerId || $inputRoot.customerId == "")
  #set($context.responseOverride.status = 400)
  #set($context.responseOverride.header.Content-Type = "application/json")
  {
    "error": "Missing required field: customerId"
  }
#else
  {
    "stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine",
    "name": "$workflowType-$util.autoId()",
    "input": "{\"order\": $util.escapeJavaScript($input.body), \"workflow_type\": \"$workflowType\"}"
  }
#end
```

**Explanation:**

* We set a default `workflowType` to `"order"`.
* If the request includes a valid `workflow_type` (like `"refund"` or `"inventory"`), we use that instead.
* The execution name and input are customized based on the workflow type.
* We still validate that `customerId` is present.

**Example Output (for a refund workflow):**

```json
{
  "stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine",
  "name": "refund-8a7b6c5d",
  "input": "{\"order\": {\"customerId\": \"67890\", \"product\": \"phone\", \"quantity\": 1, \"workflow_type\": \"refund\"}, \"workflow_type\": \"refund\"}"
}
```

This lets you use a single API endpoint to start different workflows, making your API flexible and powerful.

---

## Summary and Next Steps

In this lesson, you learned how to use API Gateway's direct integration with Step Functions to build a flexible, robust API — without needing Lambda functions. You saw how to:

* Generate unique execution names using VTL
* Validate input and return helpful error messages
* Transform Step Functions responses to be more user-friendly
* Handle errors gracefully with custom status codes and messages
* Route requests dynamically based on workflow type

You have now reached the end of this course on building serverless applications. Congratulations on making it this far! You are now ready to put these concepts into practice with the hands-on exercises that follow. Great work, and keep building!

## Generate Unique Step Function Names

Now that you understand how mapping templates work and why unique execution names are important, let's put this knowledge into practice with your first hands-on exercise.

You'll be working with an API Gateway that connects directly to Step Functions, but there's a problem — it is currently using a static execution name for all requests. This means that if two people try to start workflows at the same time, the second request will fail because Step Functions requires each execution to have a unique name.

Your job is to fix this by modifying the request mapping template to generate dynamic execution names. You need to:

* Find the TODO comment in the `template.yaml` file
* Replace the static name with a dynamic one using the `$util.autoId()` function
* Use the format `order-` followed by the auto-generated ID

This simple change will make your API much more reliable and allow multiple concurrent executions. Once you complete this, you'll have your first working direct integration between API Gateway and Step Functions!

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    {
                      "stateMachineArn": "${OrderProcessingStateMachine}",
                      # TODO: Replace the static name with a unique execution name using "order-" prefix and $util.autoId() function
                      "name": "static-order-name",
                      "input": "$util.escapeJavaScript($input.body)"
                    }
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$inputRoot.name",
                          "started_at": "$util.time.nowISO8601()",
                          "status": "RUNNING"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

The only change needed is in the `requestTemplates` mapping template: swap the hardcoded `"static-order-name"` for `"order-$util.autoId()"` (and drop the TODO comment line, since the template body is JSON). `$util.autoId()` is evaluated by API Gateway at request time, so every call gets its own execution name — and because `$` is not `${`, CloudFormation's `!Sub` leaves it untouched.

Here is the completed `template.yaml`:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    {
                      "stateMachineArn": "${OrderProcessingStateMachine}",
                      "name": "order-$util.autoId()",
                      "input": "$util.escapeJavaScript($input.body)"
                    }
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$inputRoot.name",
                          "started_at": "$util.time.nowISO8601()",
                          "status": "RUNNING"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

## Add Input Validation Logic

Excellent work on generating unique execution names! Now that your API can handle multiple concurrent requests, there's another important issue to address.

Currently, your API accepts any request and tries to start a Step Function execution, even when the request is missing critical information. This wastes resources and can lead to confusing errors later in your workflow.

Your task is to add input validation directly in the API Gateway mapping template to catch problems before they reach Step Functions. You need to:

* Find the TODO comment in the `template.yaml` file where the validation logic should be added
* Add an `#if` condition to check whether `customerId` is missing or empty
* When validation fails, return a 400 error with a helpful message
* When validation passes, proceed with starting the Step Function execution

You'll use VTL conditional logic with `#if`, `#else`, and `#end` directives to implement this validation at the API Gateway level, making your serverless application more robust without any Lambda overhead!

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    # TODO: Add validation logic here to check if customerId is missing or empty
                    # If validation fails, set response status to 400 and return error message
                    # If validation passes, proceed with Step Function execution
                    {
                      "stateMachineArn": "${OrderProcessingStateMachine}",
                      "name": "order-$util.autoId()",
                      "input": "$util.escapeJavaScript($input.body)"
                    }
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$inputRoot.name",
                          "started_at": "$util.time.nowISO8601()",
                          "status": "RUNNING"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

The change is confined to the `requestTemplates` mapping template: the three TODO comment lines get replaced with an `#if` / `#else` / `#end` block. When `customerId` is missing or empty, `$context.responseOverride.status` is set to 400 and an error body is returned instead of the `StartExecution` payload; otherwise the original payload is emitted unchanged.

Here is the completed `template.yaml`:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    #if(!$inputRoot.customerId || $inputRoot.customerId == "")
                      #set($context.responseOverride.status = 400)
                      #set($context.responseOverride.header.Content-Type = "application/json")
                      {
                        "error": "Missing required field: customerId"
                      }
                    #else
                      {
                        "stateMachineArn": "${OrderProcessingStateMachine}",
                        "name": "order-$util.autoId()",
                        "input": "$util.escapeJavaScript($input.body)"
                      }
                    #end
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$inputRoot.name",
                          "started_at": "$util.time.nowISO8601()",
                          "status": "RUNNING"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

**Example request (invalid — no `customerId`):**

```json
{"product": "laptop", "quantity": 2}
```

**Example response:**

```json
{
  "error": "Missing required field: customerId"
}
```

## Transform API Response Format

Perfect! You've successfully added input validation to prevent invalid requests from reaching your Step Functions. Now there's one more piece to make your API truly professional — the response format.

Currently, when your API successfully starts a Step Function execution, it returns the raw AWS response, which contains technical field names and lacks helpful context for your API users. The response needs to be transformed into something much more user-friendly.

Your task is to modify the response mapping template to create a clean, informative response that includes:

* Extracting execution details from the Step Functions response
* Adding a timestamp showing when the execution started
* Including a clear status indicator
* Structuring everything in an easy-to-read format

Look for the TODO comment in the response mapping template section, where you'll replace the raw response with a properly formatted JSON structure using VTL functions. This final touch will make your serverless API both robust and professional!

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    #set($workflowType = "order")
                    #if($inputRoot.workflow_type && $inputRoot.workflow_type != "")
                      #if($inputRoot.workflow_type == "order" || $inputRoot.workflow_type == "refund" || $inputRoot.workflow_type == "inventory")
                        #set($workflowType = $inputRoot.workflow_type)
                      #end
                    #end
                    #if(!$inputRoot.customerId || $inputRoot.customerId == "")
                      #set($context.responseOverride.status = 400)
                      #set($context.responseOverride.header.Content-Type = "application/json")
                      {
                        "error": "Missing required field: customerId"
                      }
                    #else
                      {
                        "stateMachineArn": "${OrderProcessingStateMachine}",
                        "name": "$workflowType-$context.requestId",
                        "input": "{\"order\": $util.escapeJavaScript($input.body), \"workflow_type\": \"$workflowType\"}"
                      }
                    #end
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        # TODO: Transform the Step Functions response into a user-friendly format
                        # Parse the response and create a JSON object with:
                        # - execution_arn from $inputRoot.executionArn
                        # - execution_name from $inputRoot.name  
                        # - started_at using $context.requestTime
                        # - status set to "RUNNING"
                        $input.body
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

The only edit is inside the `default` response's `responseTemplates`: the TODO comments and the bare `$input.body` passthrough get replaced with a `#set` that parses the Step Functions response, followed by the four-field JSON object the TODO describes. Note this exercise asks for `$context.requestTime` as the timestamp (not `$util.time.nowISO8601()` as in the lesson example).

Here is the completed `template.yaml`:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    #set($workflowType = "order")
                    #if($inputRoot.workflow_type && $inputRoot.workflow_type != "")
                      #if($inputRoot.workflow_type == "order" || $inputRoot.workflow_type == "refund" || $inputRoot.workflow_type == "inventory")
                        #set($workflowType = $inputRoot.workflow_type)
                      #end
                    #end
                    #if(!$inputRoot.customerId || $inputRoot.customerId == "")
                      #set($context.responseOverride.status = 400)
                      #set($context.responseOverride.header.Content-Type = "application/json")
                      {
                        "error": "Missing required field: customerId"
                      }
                    #else
                      {
                        "stateMachineArn": "${OrderProcessingStateMachine}",
                        "name": "$workflowType-$context.requestId",
                        "input": "{\"order\": $util.escapeJavaScript($input.body), \"workflow_type\": \"$workflowType\"}"
                      }
                    #end
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$inputRoot.name",
                          "started_at": "$context.requestTime",
                          "status": "RUNNING"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

**Example response:**

```json
{
  "execution_arn": "arn:aws:states:us-east-1:123456789012:execution:OrderProcessingStateMachine:order-7f3b2c1a",
  "execution_name": "order-7f3b2c1a",
  "started_at": "01/Jun/2024:12:34:56 +0000",
  "status": "RUNNING"
}
```

## Handle Step Functions Error Responses

Fantastic work on building a solid API with validation and response transformation! You've successfully created a serverless application that handles requests properly, but there's one final piece missing to make it truly production-ready.

Right now, your API handles basic validation errors well, but what happens when Step Functions itself encounters problems? For instance, if someone tries to start an execution with a name that already exists, or if there's an issue with the state machine itself, users receive only generic error messages that don't help them understand what went wrong.

Your task is to enhance the error handling by adding specific response patterns for common Step Functions errors. You need to:

* Add a response pattern for `ExecutionAlreadyExists` errors that returns a 409 Conflict status
* Add a response pattern for `StateMachineDoesNotExist` errors that returns a 404 Not Found status
* Add a response pattern for `InvalidParameterValue` errors that returns a 400 Bad Request status
* Enhance the existing generic error responses with more helpful information

Look for the TODO comments in the `template.yaml` file where you'll add these specific error response patterns. Each pattern should map AWS error types to appropriate HTTP status codes and provide clear, user-friendly error messages that help developers understand and fix their requests!

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::HttpApi
    Properties:
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    #set($workflowType = "order")
                    #if($inputRoot.workflow_type && $inputRoot.workflow_type != "")
                      #if($inputRoot.workflow_type == "order" || $inputRoot.workflow_type == "refund" || $inputRoot.workflow_type == "inventory")
                        #set($workflowType = $inputRoot.workflow_type)
                      #end
                    #end
                    #if(!$inputRoot.customerId || $inputRoot.customerId == "")
                      #set($context.responseOverride.status = 400)
                      #set($context.responseOverride.header.Content-Type = "application/json")
                      {
                        "error": "Missing required field: customerId"
                      }
                    #else
                      {
                        "stateMachineArn": "${OrderProcessingStateMachine}",
                        "name": "$workflowType-$util.autoId()",
                        "input": "{\"order\": $util.escapeJavaScript($input.body), \"workflow_type\": \"$workflowType\"}"
                      }
                    #end
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$inputRoot.name",
                          "started_at": "$util.time.nowISO8601()",
                          "status": "RUNNING"
                        }
                  # TODO: Add ExecutionAlreadyExists response pattern with 409 status code and user-friendly error message
                  # TODO: Add StateMachineDoesNotExist response pattern with 404 status code and clear error message
                  # TODO: Add InvalidParameterValue response pattern with 400 status code and parameter error details
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

Integration `responses` keys are **selection patterns** — regexes matched against the error text coming back from the AWS service. That means the three new entries are written as `".*ErrorName.*"` and, crucially, must sit **before** the catch-all `4\d{2}` / `5\d{2}` patterns (exactly where the TODOs are), otherwise the generic patterns would swallow them first. The last requirement is handled by adding a `message` field to the two generic responses.

Here is the completed `template.yaml`:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::HttpApi
    Properties:
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    #set($workflowType = "order")
                    #if($inputRoot.workflow_type && $inputRoot.workflow_type != "")
                      #if($inputRoot.workflow_type == "order" || $inputRoot.workflow_type == "refund" || $inputRoot.workflow_type == "inventory")
                        #set($workflowType = $inputRoot.workflow_type)
                      #end
                    #end
                    #if(!$inputRoot.customerId || $inputRoot.customerId == "")
                      #set($context.responseOverride.status = 400)
                      #set($context.responseOverride.header.Content-Type = "application/json")
                      {
                        "error": "Missing required field: customerId"
                      }
                    #else
                      {
                        "stateMachineArn": "${OrderProcessingStateMachine}",
                        "name": "$workflowType-$util.autoId()",
                        "input": "{\"order\": $util.escapeJavaScript($input.body), \"workflow_type\": \"$workflowType\"}"
                      }
                    #end
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$inputRoot.name",
                          "started_at": "$util.time.nowISO8601()",
                          "status": "RUNNING"
                        }
                  ".*ExecutionAlreadyExists.*":
                    statusCode: 409
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Execution already exists",
                          "message": "An execution with this name is already running. Please retry the request to generate a new execution name."
                        }
                  ".*StateMachineDoesNotExist.*":
                    statusCode: 404
                    responseTemplates:
                      application/json: |
                        {
                          "error": "State machine not found",
                          "message": "The target state machine does not exist. Verify the state machine ARN and that the stack is deployed."
                        }
                  ".*InvalidParameterValue.*":
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid parameter value",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '404':
                  description: State machine not found
                '409':
                  description: Execution already exists
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/start-order"
  StateMachineArn:
    Value: !Ref OrderProcessingStateMachine
```

**Example response (duplicate execution name → 409):**

```json
{
  "error": "Execution already exists",
  "message": "An execution with this name is already running. Please retry the request to generate a new execution name."
}
```

**Example response (missing state machine → 404):**

```json
{
  "error": "State machine not found",
  "message": "The target state machine does not exist. Verify the state machine ARN and that the stack is deployed."
}
```

## Implement Dynamic Workflow Routing

Excellent progress on building a robust API with validation and response transformation! You have now mastered the core concepts of direct Step Functions integration, and it is time to add the final piece that will make your API truly flexible and powerful.

Currently, your API can handle only one type of workflow, but real-world applications often need to support multiple business processes. What if you want to handle orders, refunds, and inventory updates all through the same endpoint?

Your task is to implement dynamic workflow routing by enhancing the request mapping template to read a `workflow_type` parameter from incoming requests. You need to:

* Set up a default workflow type of `"order"` for backward compatibility
* Extract and validate the `workflow_type` parameter against allowed values: `"order"`, `"refund"`, and `"inventory"`
* Use the `workflow_type` to create dynamic execution names
* Structure the Step Function input to include both the original request data and `workflow_type` information

This will transform your single-purpose API into a flexible routing system that can handle multiple workflow types through one endpoint. You will discover how VTL's conditional logic can create sophisticated routing without any Lambda overhead!

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    # TODO: Set up a default workflow type variable to "order"
                    # TODO: Add conditional logic to extract workflow_type from request body if it exists and is not empty
                    # TODO: Validate that workflow_type is one of: "order", "refund", or "inventory"
                    #if(!$inputRoot.customerId || $inputRoot.customerId == "")
                      #set($context.responseOverride.status = 400)
                      #set($context.responseOverride.header.Content-Type = "application/json")
                      {
                        "error": "Missing required field: customerId"
                      }
                    #else
                      {
                        "stateMachineArn": "${OrderProcessingStateMachine}",
                        # TODO: Use the workflow type variable in the execution name instead of hardcoded "order"
                        "name": "order-$context.requestId",
                        # TODO: Structure the input to include both the original request as "order" and the workflow_type
                        "input": "$util.escapeJavaScript($input.body)"
                      }
                    #end
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        #set($executionName = $inputRoot.executionArn.split(":")[7])
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$executionName",
                          "started_at": "$context.requestTime",
                          "status": "RUNNING"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Description: "API Gateway endpoint URL"
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Description: "Step Functions State Machine ARN"
    Value: !Ref OrderProcessingStateMachine
```

All four TODOs live in the same request mapping template. `$workflowType` starts as `"order"` (so old clients keep working), then gets overwritten only when the request supplies a `workflow_type` that passes the allow-list check — an unknown value silently falls back to `"order"` rather than erroring. The variable is then used both in the execution name and in the nested `input` payload.

Here is the completed `template.yaml`:

```yaml
AWSTemplateFormatVersion: '2010-09-09'
Transform: AWS::Serverless-2016-10-31
Description: Direct API Gateway integration with Step Functions

Resources:
  # Step Functions State Machine
  OrderProcessingStateMachine:
    Type: AWS::Serverless::StateMachine
    Properties:
      DefinitionUri: state_machine.json
      Role: !GetAtt StepFunctionExecutionRole.Arn

  # API Gateway with direct Step Functions integration
  OrderProcessingApi:
    Type: AWS::Serverless::Api
    Properties:
      StageName: Prod
      DefinitionBody:
        openapi: '3.0.1'
        info:
          title: Order Processing API
          version: '1.0'
        paths:
          /start-order:
            post:
              x-amazon-apigateway-integration:
                type: aws
                httpMethod: POST
                uri: !Sub "arn:aws:apigateway:${AWS::Region}:states:action/StartExecution"
                credentials: !GetAtt ApiGatewayStepFunctionsRole.Arn
                requestTemplates:
                  application/json: !Sub |
                    #set($inputRoot = $util.parseJson($input.body))
                    #set($workflowType = "order")
                    #if($inputRoot.workflow_type && $inputRoot.workflow_type != "")
                      #if($inputRoot.workflow_type == "order" || $inputRoot.workflow_type == "refund" || $inputRoot.workflow_type == "inventory")
                        #set($workflowType = $inputRoot.workflow_type)
                      #end
                    #end
                    #if(!$inputRoot.customerId || $inputRoot.customerId == "")
                      #set($context.responseOverride.status = 400)
                      #set($context.responseOverride.header.Content-Type = "application/json")
                      {
                        "error": "Missing required field: customerId"
                      }
                    #else
                      {
                        "stateMachineArn": "${OrderProcessingStateMachine}",
                        "name": "$workflowType-$context.requestId",
                        "input": "{\"order\": $util.escapeJavaScript($input.body), \"workflow_type\": \"$workflowType\"}"
                      }
                    #end
                responses:
                  default:
                    statusCode: 202
                    responseTemplates:
                      application/json: |
                        #set($inputRoot = $util.parseJson($input.body))
                        #set($executionName = $inputRoot.executionArn.split(":")[7])
                        {
                          "execution_arn": "$inputRoot.executionArn",
                          "execution_name": "$executionName",
                          "started_at": "$context.requestTime",
                          "status": "RUNNING"
                        }
                  4\d{2}:
                    statusCode: 400
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Invalid request parameters"
                        }
                  5\d{2}:
                    statusCode: 500
                    responseTemplates:
                      application/json: |
                        {
                          "error": "Internal server error",
                          "message": "$util.escapeJavaScript($input.body)"
                        }
              responses:
                '202':
                  description: Execution started successfully
                '400':
                  description: Bad request
                '500':
                  description: Internal server error

  # IAM Role for API Gateway to call Step Functions
  ApiGatewayStepFunctionsRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: apigateway.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionsStartExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - states:StartExecution
                Resource: !Ref OrderProcessingStateMachine

  # IAM Role for Step Functions
  StepFunctionExecutionRole:
    Type: AWS::IAM::Role
    Properties:
      AssumeRolePolicyDocument:
        Version: '2012-10-17'
        Statement:
          - Effect: Allow
            Principal:
              Service: states.amazonaws.com
            Action: sts:AssumeRole
      Policies:
        - PolicyName: StepFunctionExecutionPolicy
          PolicyDocument:
            Version: '2012-10-17'
            Statement:
              - Effect: Allow
                Action:
                  - logs:CreateLogGroup
                  - logs:CreateLogStream
                  - logs:PutLogEvents
                Resource: "*"

Outputs:
  ApiUrl:
    Description: "API Gateway endpoint URL"
    Value: !Sub "https://${OrderProcessingApi}.execute-api.${AWS::Region}.amazonaws.com/Prod/start-order"
  StateMachineArn:
    Description: "Step Functions State Machine ARN"
    Value: !Ref OrderProcessingStateMachine
```

**Example request (refund workflow):**

```json
{"customerId": "67890", "product": "phone", "quantity": 1, "workflow_type": "refund"}
```

**Resulting `StartExecution` payload:**

```json
{
  "stateMachineArn": "arn:aws:states:us-east-1:123456789012:stateMachine:OrderProcessingStateMachine",
  "name": "refund-8a7b6c5d-1234-5678-9abc-def012345678",
  "input": "{\"order\": {\"customerId\": \"67890\", \"product\": \"phone\", \"quantity\": 1, \"workflow_type\": \"refund\"}, \"workflow_type\": \"refund\"}"
}
```

A request with no `workflow_type` (or an unrecognized one) still produces `order-<requestId>`, keeping existing clients working.